In [1]:
"""
Исправленный baseline тест с правильной обработкой ListToolsResult
"""

import asyncio
import nest_asyncio
import json
import subprocess
import sys
from typing import Any, Dict, List, Optional
from contextlib import AsyncExitStack

nest_asyncio.apply()

# MCP клиент
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# LangChain
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage

# Конфигурация
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPEN_API_KEY_VSE_LLM")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.vsellm.ru/v1")
MODEL_ID = os.getenv("MODEL_ID", "deepseek/deepseek-v3.2")

# ==================== MCP КЛИЕНТ ====================

class MCPMathClient:
    """Клиент для подключения к MCP Math Server"""
    
    def __init__(self, server_script_path: str):
        self.server_script_path = server_script_path
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.tools: List[Any] = []
        self._stdio = None
        self._write = None
        
    async def connect(self, timeout: int = 30):
        """Подключение к MCP серверу"""
        print(f"🔌 Подключаемся к MCP серверу...")
        print(f"   Путь: {self.server_script_path}")
        
        if not os.path.exists(self.server_script_path):
            raise FileNotFoundError(f"MCP сервер не найден: {self.server_script_path}")
        
        server_params = StdioServerParameters(
            command=sys.executable,
            args=[self.server_script_path, "--stdio"],
            env={**os.environ, "PYTHONUNBUFFERED": "1"}
        )
        
        try:
            print("   ⏳ Ожидание соединения...")
            
            stdio_transport = await asyncio.wait_for(
                self.exit_stack.enter_async_context(stdio_client(server_params)),
                timeout=timeout
            )
            self._stdio, self._write = stdio_transport
            
            print("   ✅ Stdio транспорт установлен")
            
            self.session = await self.exit_stack.enter_async_context(
                ClientSession(self._stdio, self._write)
            )
            print("   ✅ Сессия создана")
            
            print("   ⏳ Инициализация...")
            await asyncio.wait_for(self.session.initialize(), timeout=timeout)
            print("   ✅ Инициализация успешна")
            
            # Получаем инструменты - это объект с полем tools
            print("   ⏳ Получение списка инструментов...")
            tools_result = await asyncio.wait_for(
                self.session.list_tools(), 
                timeout=timeout
            )
            
            # Извлекаем список инструментов из результата
            if hasattr(tools_result, 'tools'):
                self.tools = tools_result.tools
            elif isinstance(tools_result, list):
                self.tools = tools_result
            else:
                # Пробуем преобразовать в список
                self.tools = list(tools_result) if tools_result else []
            
            print(f"   ✅ Доступно инструментов: {len(self.tools)}")
            
            # Выводим доступные инструменты
            for tool in self.tools:
                desc = getattr(tool, 'description', 'No description')[:60]
                print(f"      📐 {tool.name}: {desc}...")
                
            return self
            
        except asyncio.TimeoutError:
            raise RuntimeError(f"Таймаут подключения к MCP серверу ({timeout}s)")
        except Exception as e:
            raise RuntimeError(f"Ошибка подключения к MCP: {str(e)}")
    
    async def call_tool(self, tool_name: str, arguments: Dict[str, Any], timeout: int = 30) -> Any:
        """Вызов инструмента MCP"""
        if not self.session:
            raise RuntimeError("MCP клиент не подключен")
        
        print(f"   🔧 Вызываем: {tool_name}({arguments})")
        
        try:
            result = await asyncio.wait_for(
                self.session.call_tool(tool_name, arguments),
                timeout=timeout
            )
            
            # Извлекаем текстовый результат
            if result.content and len(result.content) > 0:
                text_content = result.content[0].text
                try:
                    parsed = json.loads(text_content)
                    print(f"   📥 Результат: {parsed}")
                    return parsed
                except json.JSONDecodeError:
                    print(f"   📥 Результат (raw): {text_content[:100]}")
                    return {"result": text_content}
                    
            return {"result": str(result.content)}
            
        except asyncio.TimeoutError:
            raise RuntimeError(f"Таймаут вызова инструмента {tool_name}")
        except Exception as e:
            raise RuntimeError(f"Ошибка вызова {tool_name}: {e}")
    
    async def close(self):
        """Закрытие соединения"""
        try:
            await self.exit_stack.aclose()
            print("🔌 Соединение закрыто")
        except Exception as e:
            print(f"⚠️ Ошибка при закрытии: {e}")

# ==================== УПРОЩЕННЫЙ АГЕНТ ====================

class SimpleMCPAgent:
    """Упрощенный агент для тестирования MCP"""
    
    def __init__(self, mcp_client: MCPMathClient):
        self.mcp_client = mcp_client
        self.llm = ChatOpenAI(
            model=MODEL_ID,
            temperature=0,
            api_key=OPENAI_API_KEY,
            base_url=OPENAI_BASE_URL,
        )
        
    def _create_tools_prompt(self) -> str:
        """Создаем описание инструментов"""
        tools_text = []
        for tool in self.mcp_client.tools:
            params = []
            schema = getattr(tool, 'inputSchema', {}) or {}
            if 'properties' in schema:
                for name, info in schema['properties'].items():
                    ptype = info.get('type', 'any')
                    params.append(f"{name}: {ptype}")
            
            desc = getattr(tool, 'description', 'No description')
            tools_text.append(
                f"- {tool.name}({', '.join(params)}): {desc[:80]}..."
            )
        
        return "\n".join(tools_text)
    
    async def solve(self, problem: str, max_iterations: int = 3) -> Dict:
        """Решение задачи"""
        
        system_msg = f"""Ты математический ассистент. У тебя есть инструменты:

{self._create_tools_prompt()}

Инструкции:
1. Для вызова инструмента напиши: TOOL: {{"name": "имя", "args": {{...}}}}
2. Получи результат и дай финальный ответ: ANSWER: ...

Пример:
Пользователь: Реши x+2=4
Ты: TOOL: {{"name": "solve_equation", "args": {{"equation": "x+2=4"}}}}
Система: {{"solutions": "[2]"}}
Ты: ANSWER: x = 2"""

        messages = [
            SystemMessage(content=system_msg),
            HumanMessage(content=f"Задача: {problem}")
        ]
        
        history = []
        
        for i in range(max_iterations):
            print(f"\n   🔄 Шаг {i+1}/{max_iterations}")
            
            # Получаем ответ от LLM
            response = await self.llm.ainvoke(messages)
            content = response.content.strip()
            print(f"   🤖 LLM: {content[:150]}...")
            
            # Проверяем на инструмент
            if "TOOL:" in content:
                try:
                    # Извлекаем JSON
                    json_str = content.split("TOOL:")[1].strip()
                    # Берем первую строку если есть переносы
                    if "\n" in json_str:
                        json_str = json_str.split("\n")[0]
                    
                    call = json.loads(json_str)
                    tool_name = call.get("name") or call.get("tool")
                    args = call.get("args") or call.get("arguments", {})
                    
                    # Вызываем инструмент
                    result = await self.mcp_client.call_tool(tool_name, args)
                    history.append({"tool": tool_name, "args": args, "result": result})
                    
                    # Добавляем в контекст
                    messages.append(AIMessage(content=content))
                    messages.append(SystemMessage(content=f"Результат: {json.dumps(result)}"))
                    
                except Exception as e:
                    print(f"   ❌ Ошибка: {e}")
                    messages.append(AIMessage(content=content))
                    messages.append(SystemMessage(content=f"Ошибка: {e}"))
                    
            elif "ANSWER:" in content:
                answer = content.split("ANSWER:")[1].strip()
                print(f"   ✅ Ответ: {answer}")
                return {
                    "answer": answer,
                    "history": history,
                    "steps": i + 1
                }
            else:
                # Нет ни инструмента, ни ответа
                messages.append(AIMessage(content=content))
                messages.append(SystemMessage(content="Используй TOOL: для инструмента или ANSWER: для ответа"))
        
        return {"answer": "Не удалось решить", "history": history, "steps": max_iterations}

# ==================== ТЕСТЫ ====================

TEST_PROBLEMS = [
    {
        "id": 1,
        "name": "Квадратное уравнение",
        "problem": "Реши уравнение: x**2 - 5*x + 6 = 0",
        "tool": "solve_equation"
    },
    {
        "id": 2,
        "name": "Производная",
        "problem": "Найди производную от x**3",
        "tool": "differentiate"
    },
    {
        "id": 3,
        "name": "Интеграл",
        "problem": "Вычисли интеграл от 2*x",
        "tool": "integrate"
    },
    {
        "id": 4,
        "name": "Среднее",
        "problem": "Найди среднее значение чисел 10, 20, 30",
        "tool": "mean"
    },
    {
        "id": 5,
        "name": "Вычисление",
        "problem": "Вычисли sqrt(16) + 5",
        "tool": "calculate"
    }
]

async def run_tests():
    """Основной тест"""
    
    server_path = os.path.join(os.getcwd(), "calculator_server.py")
    
    print("=" * 60)
    print("🧮 BASELINE ТЕСТ MCP + LANGGRAPH")
    print("=" * 60)
    
    if not os.path.exists(server_path):
        print(f"❌ Сервер не найден: {server_path}")
        return
    
    # Подключаемся
    client = MCPMathClient(server_path)
    
    try:
        await client.connect(timeout=30)
        
        # Создаем агента
        agent = SimpleMCPAgent(client)
        
        results = []
        
        # Тестируем
        for test in TEST_PROBLEMS:
            print(f"\n{'='*60}")
            print(f"📝 Тест #{test['id']}: {test['name']}")
            print(f"Задача: {test['problem']}")
            
            try:
                result = await agent.solve(test['problem'])
                print(f"\n📊 Результат: {result['answer']}")
                
                results.append({
                    "test": test,
                    "success": len(result['history']) > 0,
                    "tools_used": [h['tool'] for h in result['history']],
                    "answer": result['answer']
                })
                
            except Exception as e:
                print(f"❌ Ошибка: {e}")
                results.append({
                    "test": test,
                    "success": False,
                    "error": str(e)
                })
        
        # Итог
        print(f"\n{'='*60}")
        print("📈 ИТОГО")
        print("=" * 60)
        
        passed = sum(1 for r in results if r.get('success'))
        print(f"✅ Успешно: {passed}/{len(TEST_PROBLEMS)}")
        
        for r in results:
            status = "✅" if r.get('success') else "❌"
            print(f"{status} {r['test']['name']}: {r.get('tools_used', [])}")
                
    except Exception as e:
        print(f"\n❌ Критическая ошибка: {e}")
        import traceback
        traceback.print_exc()
    finally:
        await client.close()

# ==================== ЗАПУСК ====================

def main():
    """Запуск в Jupyter"""
    loop = asyncio.get_event_loop()
    loop.run_until_complete(run_tests())

if __name__ == "__main__":
    main()

/home/tas/.cache/pypoetry/virtualenvs/ai-mas-hse-project-VtkJAIkS-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🧮 BASELINE ТЕСТ MCP + LANGGRAPH
🔌 Подключаемся к MCP серверу...
   Путь: /home/tas/hse-master-project/ai-mas-hse-project/ai_mas_hse_project/notebooks/calculator_server.py
   ⏳ Ожидание соединения...
   ✅ Stdio транспорт установлен
   ✅ Сессия создана
   ⏳ Инициализация...
   ✅ Инициализация успешна
   ⏳ Получение списка инструментов...
   ✅ Доступно инструментов: 23
      📐 calculate: 
    Evaluates a mathematical expression and returns the res...
      📐 solve_equation: 
    Solves an algebraic equation for x and returns all solu...
      📐 differentiate: 
    Computes the derivative of a mathematical expression wi...
      📐 integrate: 
    Computes the indefinite integral of a mathematical expr...
      📐 mean: 
    Computes the mean of a list of numbers.

    Args:
    ...
      📐 variance: 
    Computes the variance of a list of numbers.

    Args:
...
      📐 standard_deviation: 
    Computes the standard deviation of a list of numbers.

...
      📐 median: 
    Computes the medi